In [1]:
!pip install pyspark

In [2]:
import pandas as pd

data = {
    "transaction_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "customer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "product": [
        "Laptop", "Mobile", "Tablet", "Laptop", "Headphones",
        "Mobile", "Tablet", "Laptop", "Headphones", "Mobile"
    ],
    "amount": [50000, 20000, 15000, 60000, 3000,
               25000, 18000, 55000, 4000, 22000],
    "region": [
        "South", "North", "East", "South", "West",
        "North", "East", "South", "West", "North"
    ]
}

df = pd.DataFrame(data)

df.to_csv("sales_data.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [3]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("SalesReport") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession created successfully!")

# Load CSV file
sales_df = spark.read.csv(
    "sales_data.csv",
    header=True,
    inferSchema=True
)

# Display data
sales_df.show()

# Display schema
sales_df.printSchema()

SparkSession created successfully!
+--------------+-----------+----------+------+------+
|transaction_id|customer_id|   product|amount|region|
+--------------+-----------+----------+------+------+
|             1|        101|    Laptop| 50000| South|
|             2|        102|    Mobile| 20000| North|
|             3|        103|    Tablet| 15000|  East|
|             4|        104|    Laptop| 60000| South|
|             5|        105|Headphones|  3000|  West|
|             6|        106|    Mobile| 25000| North|
|             7|        107|    Tablet| 18000|  East|
|             8|        108|    Laptop| 55000| South|
|             9|        109|Headphones|  4000|  West|
|            10|        110|    Mobile| 22000| North|
+--------------+-----------+----------+------+------+

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = tr

In [4]:
# Filter sales greater than 20000
filtered_df = sales_df.filter(sales_df.amount > 20000)

# Display filtered records
filtered_df.show()

# Count the filtered records
total_records = filtered_df.count()

print("Total filtered transactions:", total_records)

+--------------+-----------+-------+------+------+
|transaction_id|customer_id|product|amount|region|
+--------------+-----------+-------+------+------+
|             1|        101| Laptop| 50000| South|
|             4|        104| Laptop| 60000| South|
|             6|        106| Mobile| 25000| North|
|             8|        108| Laptop| 55000| South|
|            10|        110| Mobile| 22000| North|
+--------------+-----------+-------+------+------+

Total filtered transactions: 5


In [5]:
from pyspark.sql.functions import sum, count, avg

# Group sales by region
region_sales = sales_df.groupBy("region").agg(
    sum("amount").alias("total_sales"),
    count("transaction_id").alias("total_transactions"),
    avg("amount").alias("average_sales")
)

# Display final results
region_sales.show()

+------+-----------+------------------+------------------+
|region|total_sales|total_transactions|     average_sales|
+------+-----------+------------------+------------------+
| South|     165000|                 3|           55000.0|
|  East|      33000|                 2|           16500.0|
|  West|       7000|                 2|            3500.0|
| North|      67000|                 3|22333.333333333332|
+------+-----------+------------------+------------------+



In [6]:
# Save aggregated data as Parquet
region_sales.write \
    .mode("overwrite") \
    .parquet("sales_report_parquet")

print("Sales report successfully saved as Parquet!")

Sales report successfully saved as Parquet!


In [7]:
import os

print("Generated Parquet files:")

for file in os.listdir("sales_report_parquet"):
    print(file)

Generated Parquet files:
.part-00000-791873bb-c755-4ee8-8cc0-156cc86d4ca1-c000.snappy.parquet.crc
_SUCCESS
part-00000-791873bb-c755-4ee8-8cc0-156cc86d4ca1-c000.snappy.parquet
._SUCCESS.crc
